# `.backward()` semantics: grad accumulation, graph freeing, `retain_graph`

Walkthrough for the autograd-mechanics file ([`../03_autograd_mechanics.md`](../03_autograd_mechanics.md)). Four small experiments on a trivial graph `y = x.sum()`, each with a **prediction first**, then the code, then what actually happened and why.

Why `sum`? Because `d(sum)/dx_i = 1` for every element, so the gradient is a clean vector of ones. That makes accumulation easy to read: one backward gives `[1,1,1]`, two give `[2,2,2]`. Nothing about the math is hidden — the experiments are purely about *autograd's bookkeeping*: which tensors get a graph, when the graph is freed, and how `.grad` accumulates.

In [ ]:
import torch
torch.manual_seed(0)  # deterministic x; grads don't depend on the values here anyway

## Case 1 — the baseline

```python
x = torch.randn(3, requires_grad=True); y = x.sum(); y.backward(); print(x.grad)
```

**Prediction:** `x.grad == tensor([1., 1., 1.])`.

`x` is a leaf with `requires_grad=True`. `y = x.sum()` builds a graph (`y` gets a `SumBackward` `grad_fn`). `y.backward()` seeds `dy/dy = 1` and pushes it back through `sum`, whose local gradient w.r.t. each input element is `1`. Result lands in the leaf's `.grad`.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x.sum()
print("y.grad_fn:", y.grad_fn)        # <SumBackward0> — y is an intermediate
y.backward()
print("x.grad:", x.grad)              # tensor([1., 1., 1.])

**Result:** `x.grad = tensor([1., 1., 1.])`, and `y.grad_fn` is a `SumBackward0` node — confirming `y` is a graph-intermediate while `x` is the leaf that actually stores the gradient.

## Case 2 — no `requires_grad`

```python
x = torch.randn(3); y = x.sum(); y.backward()
```

**Prediction:** `RuntimeError` on `y.backward()`.

With `requires_grad=False` (the default), no input to `sum` requires grad, so **no graph is built**: `y.requires_grad is False` and `y.grad_fn is None`. There is nothing to back-propagate through, and autograd refuses. The message is roughly: *"element 0 of tensors does not require grad and does not have a grad_fn."*

In [ ]:
x = torch.randn(3)               # no requires_grad
y = x.sum()
print("y.requires_grad:", y.requires_grad)   # False
print("y.grad_fn:", y.grad_fn)               # None
try:
    y.backward()
except RuntimeError as e:
    print("RuntimeError:", e)

**Result:** `RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn`. The lesson: the graph only exists if *something* feeding the output requires grad. A frozen forward pass (no `requires_grad` anywhere) builds no graph and cannot be backpropagated.

## Case 3 — backward twice (graph already freed)

```python
x = torch.randn(3, requires_grad=True); y = x.sum(); y.backward(); y.backward()
```

**Prediction:** the *first* `y.backward()` succeeds; the *second* raises `RuntimeError`.

By default, backward **frees the graph** (the saved intermediate buffers) as it traverses, to reclaim memory. The second call finds the buffers gone and errors: *"Trying to backward through the graph a second time ... Specify retain_graph=True if you need to backward through the graph a second time."* Note this is about the **graph being freed**, not about gradient accumulation — `x.grad` was already populated by the first call.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x.sum()
y.backward()
print("x.grad after first backward:", x.grad)   # tensor([1., 1., 1.])
try:
    y.backward()                                 # graph was freed by the first call
except RuntimeError as e:
    print("RuntimeError:", e)

**Result:** first backward fine (`x.grad = [1,1,1]`); second raises `RuntimeError: Trying to backward through the graph a second time ...`. This is why the standard training loop is one-forward / one-backward / one-step / new-forward — each iteration rebuilds a fresh graph.

## Case 4 — `retain_graph=True`, then accumulation shows

```python
x = torch.randn(3, requires_grad=True); y = x.sum(); y.backward(retain_graph=True); y.backward(); print(x.grad)
```

**Prediction:** `x.grad == tensor([2., 2., 2.])`.

`retain_graph=True` on the first call keeps the graph alive, so the second `y.backward()` is legal (Case 3's error is gone). But `.grad` on a leaf is **additive** — autograd *adds* into `.grad` rather than overwriting. First backward writes `[1,1,1]`; second adds another `[1,1,1]` → `[2,2,2]`. This additivity is exactly why training loops call `optimizer.zero_grad()` every step: without it, gradients from old steps would pile up.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x.sum()
y.backward(retain_graph=True)        # keep the graph so we can backward again
print("x.grad after first backward:", x.grad)    # tensor([1., 1., 1.])
y.backward()                         # second backward: legal, and ACCUMULATES
print("x.grad after second backward:", x.grad)   # tensor([2., 2., 2.])

**Result:** `x.grad = tensor([2., 2., 2.])` — the two backward passes summed. Note the second call did *not* pass `retain_graph=True`, so it freed the graph; a *third* backward would now raise the Case-3 error.

### Takeaways

| Knob | Controls | Default behavior |
|------|----------|------------------|
| `requires_grad` | whether a graph is built at all | `False` → no graph, backward errors (Case 2) |
| `retain_graph` | whether saved buffers survive backward | `False` → graph freed, second backward errors (Case 3) |
| `.grad` accumulation | how repeated backward writes to leaves | always additive → must `zero_grad()` between steps (Case 4) |

Two orthogonal mechanisms get conflated constantly: **graph freeing** (a memory thing, fixed by `retain_graph`) and **gradient accumulation** (a correctness thing, fixed by `zero_grad`). Case 3 is about the first; Case 4 reveals the second.